# IntelliOptiCloud: XGBoost Memory Right-Sizing Model (v2)
FYP training notebook (Google Colab, free tier). The model saved here is the **same model the deployed backend loads**.

**Note:** training data is *synthetic* (generated from a rule plus noise) because no real labelled dataset of Lambda right-sizing outcomes is available. Accuracy measures how well the model learns this rule, not real-world savings.

In [ ]:
!pip -q install xgboost matplotlib

In [ ]:
import json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

SIZES = np.array([128, 256, 512, 1024, 2048, 3008])   # valid Lambda memory sizes used by the app
FEATURES = ["memory", "invocations", "duration", "cold_ratio", "hour", "util"]  # order MUST match backend

def snap(values):
    values = np.asarray(values, dtype=float)
    return SIZES[np.abs(values[:, None] - SIZES[None, :]).argmin(axis=1)]

## 1. Generate synthetic training data
Target rule: optimal memory = memory x utilisation x 1.2 headroom, snapped to a valid Lambda size.

In [ ]:
rng = np.random.default_rng(42)
n = 5000
memory = rng.choice(SIZES, n)
invocations = rng.integers(1_000, 500_000, n)
duration = rng.uniform(10, 3000, n)
cold_ratio = rng.uniform(0, 0.3, n)
hour = rng.integers(0, 24, n)
util = rng.uniform(0.1, 0.95, n)

X = np.column_stack([memory, invocations, duration, cold_ratio, hour, util])
y = snap(memory * util * 1.2)
pd.DataFrame(X, columns=FEATURES).assign(target=y).head()

## 2. Train / test split and training

In [ ]:
idx = rng.permutation(n)
cut = int(n * 0.8)
train, test = idx[:cut], idx[cut:]

model = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=1)
model.fit(X[train], y[train])

## 3. Evaluate

In [ ]:
pred = model.predict(X[test])
mae = float(np.mean(np.abs(pred - y[test])))
accuracy = round(max(0.0, 100 - mae / float(y[test].mean()) * 100), 1)
exact = round(float((snap(pred) == y[test]).mean() * 100), 1)
print("MAE (MB):", round(mae, 2))
print("Accuracy (1 - MAE/mean):", accuracy, "%")
print("Exact valid-size match:", exact, "%")

pd.Series(model.feature_importances_, index=FEATURES).sort_values().plot.barh(title="Feature importance")
plt.show()

## 4. Save the model and its metadata
Download both files and place them in `backend/app/model/` in the repo.

In [ ]:
model.save_model("memory_model.json")
json.dump({"accuracy": accuracy, "mae": round(mae, 2), "exact_match": exact,
           "features": FEATURES, "training_samples": int(cut), "data": "synthetic"},
          open("model_meta.json", "w"))
from google.colab import files
files.download("memory_model.json")
files.download("model_meta.json")